# LeetCode 314 — Binary Tree Vertical Order Traversal bằng Python

Link đề: https://leetcode.com/problems/binary-tree-vertical-order-traversal/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Breadth-First Search, Sorting, Binary Tree

> Đây là bài **Premium**: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def verticalOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: gán cho mỗi nút một **cột** (gốc ở cột `0`; con trái ở `cột − 1`, con phải ở `cột + 1`), rồi trả về giá trị các nút **theo từng cột từ trái sang phải**; trong cùng một cột thì **từ trên xuống dưới**; nếu hai nút cùng cột **và cùng hàng** thì nút **bên trái** đứng trước.
2. Biết tự dựng `TreeNode` và hàm `build_tree` từ danh sách theo thứ tự từng tầng (có `None`) để kiểm thử nhanh.
3. Đi từ **thu thập `(cột, hàng, vị trí)` rồi sắp xếp** → **BFS với dict `cột → danh sách`** → **DFS ghi `(độ sâu, giá trị)` rồi sắp xếp ổn định theo độ sâu** → **BFS hai lượt dùng mảng theo `min/max` cột** → **BFS theo từng tầng**.
4. Hiểu **vì sao BFS tự giữ đúng thứ tự "trên xuống, trái sang phải"** còn DFS cần thêm khóa độ sâu.
5. Tránh lỗi kinh điển: DFS mà không sắp theo độ sâu, sắp theo **giá trị** (bài 987 mới sắp theo giá trị), quên sắp các cột (dict giữ thứ tự chèn), quên cây rỗng.

## 2. Đề bài & ví dụ

Cho gốc của một cây nhị phân, trả về **duyệt theo chiều dọc (vertical order traversal)** của các giá trị nút: từ trên xuống dưới, từng cột một, từ trái sang phải. Nếu hai nút cùng hàng và cùng cột thì **nút bên trái** đứng trước.

**Ví dụ 1:** `root = [3,9,20,null,null,15,7]` → `[[9],[3,15],[20],[7]]`.

**Ví dụ 2:** `root = [3,9,8,4,0,1,7]` → `[[4],[9],[3,0,1],[8],[7]]`.

**Ví dụ 3:** `root = [3,9,8,4,0,1,7,null,null,null,2,5]` → `[[4],[9,5],[3,0,1],[8,2],[7]]`.

**Ví dụ 4:** `root = []` → `[]`.

**Ràng buộc:** số nút trong cây thuộc `[0, 100]`; `-100 <= Node.val <= 100`.

In [1]:
from typing import List, Optional
from collections import deque, defaultdict
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(vals):
    """Dựng cây từ danh sách theo từng tầng (kiểu LeetCode), None = nút trống."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            q.append(node.left)
            q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([3, 9, 20, None, None, 15, 7], [[9], [3, 15], [20], [7]]),
    ([3, 9, 8, 4, 0, 1, 7], [[4], [9], [3, 0, 1], [8], [7]]),
    ([3, 9, 8, 4, 0, 1, 7, None, None, None, 2, 5], [[4], [9, 5], [3, 0, 1], [8, 2], [7]]),
    ([], []),
]
for vals, e in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print("root =", vals, "-> kỳ vọng", e)

root = [3, 9, 20, None, None, 15, 7] -> kỳ vọng [[9], [3, 15], [20], [7]]
root = [3, 9, 8, 4, 0, 1, 7] -> kỳ vọng [[4], [9], [3, 0, 1], [8], [7]]
root = [3, 9, 8, 4, 0, 1, 7, None, None, None, 2, 5] -> kỳ vọng [[4], [9, 5], [3, 0, 1], [8, 2], [7]]
root = [] -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Cột của nút

Gốc ở cột `0`. Con trái của nút ở cột `c` có cột `c − 1`; con phải có cột `c + 1`. Các nút cùng cột nằm trên cùng một đường thẳng đứng (nhưng có thể ở nhiều độ sâu khác nhau).

In [2]:
def columns_dfs(root):
    out = []

    def go(node, col, row):
        if node is None:
            return
        out.append((node.val, col, row))
        go(node.left, col - 1, row + 1)
        go(node.right, col + 1, row + 1)

    go(root, 0, 0)
    return out


print(columns_dfs(build_tree([3, 9, 20, None, None, 15, 7])))     # (giá trị, cột, hàng)

[(3, 0, 0), (9, -1, 1), (20, 1, 1), (15, 0, 2), (7, 2, 2)]


### 3.2. Hai quy tắc phân xử khi cùng cột

- **Khác hàng:** nút ở hàng nhỏ hơn (cao hơn) đứng trước.
- **Cùng hàng:** nút bên **trái** đứng trước. Hai nút cùng hàng, cùng cột thuộc hai nhánh con khác nhau của tổ tiên chung gần nhất; nhánh trái nằm bên trái của nhánh phải.

Ghi chú: bài **987 (Vertical Order Traversal of a Binary Tree)** khác ở chỗ khi cùng hàng và cùng cột thì sắp theo **giá trị** tăng dần; bài này (314) thì theo **trái → phải**.

In [3]:
# Hai nút cùng hàng, cùng cột (ví dụ 2): 0 (con phải của 9) và 1 (con trái của 8) đều ở cột 0, hàng 2
root = build_tree([3, 9, 8, 4, 0, 1, 7])
print([t for t in columns_dfs(root) if t[1] == 0])      # 3 (hàng 0), 0 (hàng 2), 1 (hàng 2): 0 đứng trước 1 vì ở bên trái

[(3, 0, 0), (0, 0, 2), (1, 0, 2)]


### 3.3. BFS (duyệt theo tầng) bằng `deque`

BFS thăm các nút **tầng này xong mới sang tầng sau**, và trong một tầng thì từ **trái sang phải** (nhờ thêm con trái trước con phải). Vì thế thứ tự thăm của BFS chính là thứ tự "hàng trên trước, cùng hàng thì trái trước": đúng yêu cầu phân xử của bài. Chỉ cần nhóm theo cột là xong.

In [4]:
def bfs_order(root):
    out, q = [], deque([root] if root else [])
    while q:
        node = q.popleft()
        out.append(node.val)
        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
    return out


print(bfs_order(build_tree([3, 9, 8, 4, 0, 1, 7])))

[3, 9, 8, 4, 0, 1, 7]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 3: `root = [3,9,8,4,0,1,7,null,null,null,2,5]`.

Cây: `3` có con `9` (trái), `8` (phải); `9` có con `4`, `0`; `8` có con `1`, `7`; `0` có con phải `2`; `1` có con trái `5`.

| nút | cột | hàng |
|---|---|---|
| 3 | 0 | 0 |
| 9 | −1 | 1 |
| 8 | 1 | 1 |
| 4 | −2 | 2 |
| 0 | 0 | 2 |
| 1 | 0 | 2 |
| 7 | 2 | 2 |
| 2 | 1 | 3 |
| 5 | −1 | 3 |

Nhóm theo cột (theo thứ tự BFS): cột −2: `[4]`; cột −1: `[9, 5]`; cột 0: `[3, 0, 1]`; cột 1: `[8, 2]`; cột 2: `[7]`. Đọc từ cột nhỏ nhất đến lớn nhất.

In [5]:
root = build_tree([3, 9, 8, 4, 0, 1, 7, None, None, None, 2, 5])
cols = defaultdict(list)
q = deque([(root, 0)])
while q:
    node, c = q.popleft()
    cols[c].append(node.val)
    if node.left:
        q.append((node.left, c - 1))
    if node.right:
        q.append((node.right, c + 1))
print([cols[c] for c in sorted(cols)])

[[4], [9, 5], [3, 0, 1], [8, 2], [7]]


## 5. Cách 1 — Brute Force / baseline: thu thập `(cột, hàng, vị trí)` rồi sắp xếp

**Ý tưởng.** DFS gán cho mỗi nút bộ ba `(cột, hàng, vị trí)` với **vị trí** là chỉ số theo cách đánh số của cây đầy đủ (`gốc = 1`, con trái `2p`, con phải `2p + 1`); trong cùng một hàng, vị trí nhỏ hơn nghĩa là **nằm bên trái hơn**. Sắp xếp toàn bộ theo `(cột, hàng, vị trí)` rồi nhóm theo cột. Cách này **không dựa vào BFS** nên là oracle độc lập.

In [6]:
def vertical_order_brute(root: Optional[TreeNode]) -> List[List[int]]:
    items = []

    def go(node, col, row, pos):
        if node is None:
            return
        items.append((col, row, pos, node.val))
        go(node.left, col - 1, row + 1, 2 * pos)
        go(node.right, col + 1, row + 1, 2 * pos + 1)

    go(root, 0, 0, 1)
    items.sort()
    result = []
    last_col = None
    for col, row, pos, val in items:
        if col != last_col:
            result.append([])
            last_col = col
        result[-1].append(val)
    return result


for vals, e in examples:
    assert vertical_order_brute(build_tree(vals)) == e
print("PASS ví dụ (brute force)")

PASS ví dụ (brute force)


**Độ phức tạp (Cách 1).** Time `O(n log n)` do sắp xếp; Space `O(n)` (danh sách bộ ba, ngăn xếp đệ quy `O(h)`). Vị trí có thể là số rất lớn khi cây sâu, nhưng Python xử lý số nguyên lớn nên vẫn đúng.

**Nút thắt.** Phải sắp xếp trong khi BFS **đã cho sẵn** thứ tự cần thiết. Ta có thể `O(n)` (trừ việc sắp các cột).

## 6. Ý tưởng thuật toán / bước đột phá

**BFS đã thăm các nút theo đúng thứ tự "hàng trên trước, cùng hàng thì trái trước".** Vậy ta chỉ cần:

1. Cho mỗi nút trong hàng đợi kèm theo **cột** của nó.
2. Khi lấy nút ra, **nối giá trị vào danh sách của cột đó** (`dict[cột].append(val)`).
3. Ở cuối, **đọc các cột theo thứ tự tăng dần** (`sorted(dict)`, hoặc theo `min..max` cột nếu đã biết).

Không cần sắp xếp các nút, chỉ sắp **các khóa cột** (tối đa `n` khóa; với cây `n = 100` rất nhỏ).

In [7]:
# Minh họa: thứ tự thăm của BFS giữ nguyên quy tắc (hàng, vị trí) mà không cần sắp xếp
rnd = random.Random(314)


def random_tree(n, rnd, skew=0.0, lo=-100, hi=100):
    """Cây ngẫu nhiên n nút: chọn ngẫu nhiên một ô trống để gắn nút mới (skew > 0 làm cây lệch)."""
    if n == 0:
        return None
    root = TreeNode(rnd.randint(lo, hi))
    slots = [(root, 0), (root, 1)]
    for _ in range(n - 1):
        k = len(slots) - 1 if rnd.random() < skew else rnd.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        node = TreeNode(rnd.randint(lo, hi))
        if side == 0:
            parent.left = node
        else:
            parent.right = node
        slots.append((node, 0))
        slots.append((node, 1))
    return root


for _ in range(200):
    root = random_tree(rnd.randint(0, 40), rnd)
    items = []
    q = deque([(root, 0, 0, 1)] if root else [])
    while q:
        node, col, row, pos = q.popleft()
        items.append((row, pos))
        if node.left:
            q.append((node.left, col - 1, row + 1, 2 * pos))
        if node.right:
            q.append((node.right, col + 1, row + 1, 2 * pos + 1))
    assert items == sorted(items)           # BFS thăm theo (hàng, vị trí) tăng dần
print("PASS: BFS thăm theo thứ tự (hàng, vị trí) tăng dần trên 200 cây ngẫu nhiên")

PASS: BFS thăm theo thứ tự (hàng, vị trí) tăng dần trên 200 cây ngẫu nhiên


## 7. Các cách tiếp theo

### Cách 2 — BFS với dict `cột → danh sách` (cách tham khảo)

**Giải thích.** Hàng đợi chứa `(nút, cột)`. Mỗi nút lấy ra: `cols[cột].append(giá trị)`; con trái vào hàng đợi với `cột − 1`, con phải với `cột + 1`. Cuối cùng trả `[cols[c] for c in sorted(cols)]`. Cây rỗng trả `[]` ngay.

In [8]:
def vertical_order_bfs_dict(root: Optional[TreeNode]) -> List[List[int]]:
    if root is None:
        return []
    cols = defaultdict(list)
    q = deque([(root, 0)])
    while q:
        node, c = q.popleft()
        cols[c].append(node.val)
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    return [cols[c] for c in sorted(cols)]


for vals, e in examples:
    assert vertical_order_bfs_dict(build_tree(vals)) == e
print("PASS ví dụ (BFS + dict)")

PASS ví dụ (BFS + dict)


**Độ phức tạp Cách 2:** Time `O(n + w log w)` với `w` là số cột (`w ≤ n`), tức `O(n log n)` xấu nhất; Space `O(n)`.

### Cách 3 — DFS ghi `(độ sâu, giá trị)` theo cột rồi sắp ổn định theo độ sâu

**Giải thích.** DFS tiền tự (`gốc, trái, phải`) gắn `(độ sâu, giá trị)` vào `cols[cột]`. Với mỗi cột, sắp **ổn định** theo độ sâu (`sort(key=lambda t: t[0])`). Ổn định để hai nút cùng cột, cùng độ sâu giữ thứ tự DFS tiền tự, tức **trái trước phải** (hai nút này thuộc hai nhánh con khác nhau nên DFS thăm hết nhánh trái trước).

In [9]:
def vertical_order_dfs(root: Optional[TreeNode]) -> List[List[int]]:
    cols = defaultdict(list)

    def dfs(node, depth, col):
        if node is None:
            return
        cols[col].append((depth, node.val))
        dfs(node.left, depth + 1, col - 1)
        dfs(node.right, depth + 1, col + 1)

    dfs(root, 0, 0)
    return [[v for _, v in sorted(cols[c], key=lambda t: t[0])] for c in sorted(cols)]


for vals, e in examples:
    assert vertical_order_dfs(build_tree(vals)) == e
print("PASS ví dụ (DFS + sắp ổn định)")

PASS ví dụ (DFS + sắp ổn định)


**Độ phức tạp Cách 3:** Time `O(n log n)`; Space `O(n + h)` (ngăn xếp đệ quy `h ≤ 100`).

### Cách 4 — BFS hai lượt: tìm `min/max` cột rồi dùng mảng

**Giải thích.** Lượt 1 (có thể kết hợp với BFS) tìm cột nhỏ nhất `lo` và lớn nhất `hi`. Lượt 2 BFS đặt giá trị vào `buckets[cột − lo]` trong mảng `hi − lo + 1` danh sách. Không cần sắp các khóa: kết quả là mảng `buckets`. Cách này giảm `log` và rất ổn khi dùng `list` thay `dict`.

In [10]:
def vertical_order_two_pass(root: Optional[TreeNode]) -> List[List[int]]:
    if root is None:
        return []
    lo = hi = 0
    q = deque([(root, 0)])
    while q:                                   # lượt 1: tìm cột nhỏ/lớn nhất
        node, c = q.popleft()
        lo, hi = min(lo, c), max(hi, c)
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    buckets = [[] for _ in range(hi - lo + 1)]
    q = deque([(root, 0)])
    while q:                                   # lượt 2: BFS đặt giá trị vào đúng ô
        node, c = q.popleft()
        buckets[c - lo].append(node.val)
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    return buckets


for vals, e in examples:
    assert vertical_order_two_pass(build_tree(vals)) == e
print("PASS ví dụ (hai lượt BFS)")

PASS ví dụ (hai lượt BFS)


**Độ phức tạp Cách 4:** Time `O(n)` (không sắp xếp); Space `O(n)`.

### Cách 5 — BFS theo từng tầng (level by level)

**Giải thích.** Xử lý hàng đợi **theo từng tầng** (vòng `for _ in range(len(q))`). Trong mỗi tầng, các nút ở cùng hàng và được thăm từ trái sang phải. Dùng `dict` cột → danh sách. Hàng đợi tách tầng giúp mở rộng sang các biến thể cần biết hàng (bài 987).

In [11]:
def vertical_order_levels(root: Optional[TreeNode]) -> List[List[int]]:
    if root is None:
        return []
    cols = {}
    level = [(root, 0)]
    while level:
        nxt = []
        for node, c in level:                  # một tầng, từ trái sang phải
            cols.setdefault(c, []).append(node.val)
            if node.left:
                nxt.append((node.left, c - 1))
            if node.right:
                nxt.append((node.right, c + 1))
        level = nxt
    return [cols[c] for c in range(min(cols), max(cols) + 1)]


for vals, e in examples:
    assert vertical_order_levels(build_tree(vals)) == e
print("PASS ví dụ (BFS theo tầng)")

PASS ví dụ (BFS theo tầng)


**Độ phức tạp Cách 5:** Time `O(n)` (dùng `range(min, max + 1)`, các cột liên tiếp nên không cần sắp); Space `O(n)`.

> Cách 2 là cách nộp bài (ngắn, chuẩn). Cách 4 và 5 tránh sắp xếp. Cách 3 minh họa DFS. Cách 1 làm oracle.

## 8. TRACE từng bước

Theo dõi BFS (Cách 2) trên ví dụ 3: in nút lấy ra, cột, và trạng thái `cols`.

In [12]:
def trace_vertical(root):
    cols = defaultdict(list)
    q = deque([(root, 0)])
    step = 0
    while q:
        node, c = q.popleft()
        step += 1
        cols[c].append(node.val)
        print(f"bước {step}: lấy nút {node.val} ở cột {c} -> cols = {dict(sorted(cols.items()))}")
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    result = [cols[c] for c in sorted(cols)]
    print("kết quả:", result)
    return result


assert trace_vertical(build_tree([3, 9, 8, 4, 0, 1, 7, None, None, None, 2, 5])) == [[4], [9, 5], [3, 0, 1], [8, 2], [7]]

bước 1: lấy nút 3 ở cột 0 -> cols = {0: [3]}
bước 2: lấy nút 9 ở cột -1 -> cols = {-1: [9], 0: [3]}
bước 3: lấy nút 8 ở cột 1 -> cols = {-1: [9], 0: [3], 1: [8]}
bước 4: lấy nút 4 ở cột -2 -> cols = {-2: [4], -1: [9], 0: [3], 1: [8]}
bước 5: lấy nút 0 ở cột 0 -> cols = {-2: [4], -1: [9], 0: [3, 0], 1: [8]}
bước 6: lấy nút 1 ở cột 0 -> cols = {-2: [4], -1: [9], 0: [3, 0, 1], 1: [8]}
bước 7: lấy nút 7 ở cột 2 -> cols = {-2: [4], -1: [9], 0: [3, 0, 1], 1: [8], 2: [7]}
bước 8: lấy nút 2 ở cột 1 -> cols = {-2: [4], -1: [9], 0: [3, 0, 1], 1: [8, 2], 2: [7]}
bước 9: lấy nút 5 ở cột -1 -> cols = {-2: [4], -1: [9, 5], 0: [3, 0, 1], 1: [8, 2], 2: [7]}
kết quả: [[4], [9, 5], [3, 0, 1], [8, 2], [7]]


Thứ tự BFS: `3, 9, 8, 4, 0, 1, 7, 2, 5` (tầng 0: `3`; tầng 1: `9, 8`; tầng 2: `4, 0, 1, 7`; tầng 3: `2, 5`). Nút `5` (con của `1`) nằm sau `2` (con của `0`) trong BFS; nhưng chúng ở hai cột khác nhau nên không ảnh hưởng nhau: `5` thuộc cột −1, đứng sau `9` (hàng 1 < hàng 3).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (BFS + dict, `sorted` các khóa cột). LeetCode định nghĩa sẵn `TreeNode`; ở đây ta đã có lớp `TreeNode` bên trên.

In [13]:
class Solution:
    def verticalOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
        if root is None:
            return []
        cols = defaultdict(list)
        q = deque([(root, 0)])
        while q:
            node, c = q.popleft()
            cols[c].append(node.val)
            if node.left:
                q.append((node.left, c - 1))
            if node.right:
                q.append((node.right, c + 1))
        return [cols[c] for c in sorted(cols)]


for vals, e in examples:
    assert Solution().verticalOrder(build_tree(vals)) == e
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Cột được gán đúng:** truyền `cột ± 1` từ cha xuống con nên mỗi nút nhận đúng cột theo định nghĩa.
- **BFS thăm theo thứ tự (hàng, vị trí) tăng dần:** BFS thăm hết tầng `d` trước tầng `d + 1`. Trong một tầng, thứ tự thăm bằng thứ tự thêm vào hàng đợi; nếu tầng `d` được thăm từ trái sang phải thì con của chúng (thêm theo thứ tự trái rồi phải) cũng được thêm từ trái sang phải ở tầng `d + 1`. Quy nạp theo tầng ⇒ mỗi tầng thăm từ trái sang phải.
- **Nhóm theo cột giữ thứ tự:** `append` vào danh sách của cột theo thứ tự thăm nên mỗi cột có các nút theo (hàng, trái → phải): đúng yêu cầu.
- **Đọc cột theo thứ tự tăng:** `sorted(cols)` (hoặc `range(min, max + 1)`) cho thứ tự trái → phải.
- **DFS (Cách 3):** hai nút cùng cột cùng độ sâu thuộc hai nhánh con khác nhau của tổ tiên chung; DFS tiền tự thăm nhánh trái trước nên ổn định sau khi sắp theo độ sâu vẫn giữ trái trước.
- **Cây rỗng:** trả `[]`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `w` là số cột.

- **Cách 1:** Time `O(n log n)`; Space `O(n)`.
- **Cách 2:** Time `O(n + w log w)`; Space `O(n)`.
- **Cách 3:** Time `O(n log n)`; Space `O(n + h)`.
- **Cách 4:** Time `O(n)`; Space `O(n)`.
- **Cách 5:** Time `O(n)`; Space `O(n)`.
- Với `n ≤ 100`: tất cả đều tức thì; khác biệt chỉ có ý nghĩa với cây rất lớn.

## 12. Bộ test

Cây được dựng bằng `build_tree` từ danh sách tầng. Mọi cách so với **oracle brute force** (`(cột, hàng, vị trí)` rồi sắp xếp), không dựa vào BFS. Kết quả là danh sách cố định (có một đáp án duy nhất) nên so sánh bằng `==`.

In [14]:
impls = {
    "brute": vertical_order_brute,
    "bfs_dict": vertical_order_bfs_dict,
    "dfs": vertical_order_dfs,
    "two_pass": vertical_order_two_pass,
    "levels": vertical_order_levels,
    "submit": lambda root: Solution().verticalOrder(root),
}


def check_tree(root, expected=None):
    base = vertical_order_brute(root)
    if expected is not None:
        assert base == expected, (tree_to_list(root), base, expected)
    for name, fn in impls.items():
        got = fn(root)
        assert got == base, (name, tree_to_list(root), got, base)
    return base


def check(vals, expected=None):
    return check_tree(build_tree(vals), expected)


for vals, e in examples:
    check(vals, e)
print("PASS 4 ví dụ của đề")

PASS 4 ví dụ của đề


### 12.1. Các trường hợp biên và giá trị tính tay

In [15]:
cases = [
    ([], [], "cây rỗng"),
    ([1], [[1]], "một nút"),
    ([1, 2, 3], [[2], [1], [3]], "ba nút"),
    ([1, 2], [[2], [1]], "chỉ con trái"),
    ([1, None, 2], [[1], [2]], "chỉ con phải"),
    ([1, 2, 3, 4, 6, 5, 7], [[4], [2], [1, 6, 5], [3], [7]], "nhiều nút cùng cột cùng hàng: 6 đứng trước 5"),
    ([1, 2, 3, None, 4, 5], [[2], [1, 4, 5], [3]], "cột 0 gồm 1, 4, 5"),
    ([1, 2, None, 3, None, 4], [[4], [3], [2], [1]], "lệch trái (chuỗi con trái)"),
    ([1, None, 2, None, 3, None, 4], [[1], [2], [3], [4]], "lệch phải (chuỗi con phải)"),
    ([5, 5, 5, 5, 5, 5, 5], [[5], [5], [5, 5, 5], [5], [5]], "giá trị trùng nhau"),
    ([-100, 100, -100], [[100], [-100], [-100]], "giá trị biên +-100"),
    ([0, 0, 0, 0, None, None, 0, None, 0], [[0, 0], [0], [0, 0], [0]], "toàn số 0 (kết quả cần kiểm tra bằng brute)"),
]
for vals, e, why in cases:
    check(vals, e if "kiểm tra bằng brute" not in why else None)
    print("PASS", why)

PASS cây rỗng
PASS một nút
PASS ba nút
PASS chỉ con trái
PASS chỉ con phải
PASS nhiều nút cùng cột cùng hàng: 6 đứng trước 5
PASS cột 0 gồm 1, 4, 5
PASS lệch trái (chuỗi con trái)
PASS lệch phải (chuỗi con phải)
PASS giá trị trùng nhau
PASS giá trị biên +-100
PASS toàn số 0 (kết quả cần kiểm tra bằng brute)


### 12.2. Tính chất và phản ví dụ

In [16]:
rnd = random.Random(314)

# Tính chất 1: các danh sách chứa đúng đa tập giá trị của cây (không mất, không thừa nút)
# Tính chất 2: số cột = (cột lớn nhất - cột nhỏ nhất + 1); mọi cột không rỗng
# Tính chất 3: lật gương cây (đổi trái <-> phải) cho kết quả với thứ tự cột đảo, còn thứ tự trong cột: trên->dưới giữ nguyên, cùng hàng đảo ngược
def mirror(root):
    if root is None:
        return None
    return TreeNode(root.val, mirror(root.right), mirror(root.left))


for _ in range(300):
    root = random_tree(rnd.randint(0, 40), rnd, skew=rnd.choice([0, 0.5, 0.9]))
    res = vertical_order_bfs_dict(root)
    flat = sorted(v for col in res for v in col)
    nodes = sorted(v for v, _, _ in columns_dfs(root))
    assert flat == nodes
    cols_used = {c for _, c, _ in columns_dfs(root)}
    assert len(res) == (max(cols_used) - min(cols_used) + 1 if cols_used else 0)
    assert all(col for col in res)
    # lật gương: danh sách cột đảo ngược
    mir = vertical_order_bfs_dict(mirror(root))
    assert [sorted(c) for c in mir] == [sorted(c) for c in reversed(res)]
print("PASS tính chất: bảo toàn giá trị, số cột liên tiếp, đối xứng khi lật gương (so theo tập từng cột)")

# Phản ví dụ: cùng hàng cùng cột thì TRÁI trước (khác bài 987: theo giá trị)
root = build_tree([1, 2, 3, 4, 6, 5, 7])
assert vertical_order_bfs_dict(root)[2] == [1, 6, 5]          # 6 (nhánh trái) trước 5 (nhánh phải) dù 5 < 6
print("PASS phản ví dụ: [1, 6, 5] (trái -> phải), không phải [1, 5, 6]")

PASS tính chất: bảo toàn giá trị, số cột liên tiếp, đối xứng khi lật gương (so theo tập từng cột)
PASS phản ví dụ: [1, 6, 5] (trái -> phải), không phải [1, 5, 6]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [17]:
rnd = random.Random(2026)
for trial in range(1000):
    n = rnd.randint(0, 100)
    check_tree(random_tree(n, rnd, skew=rnd.choice([0, 0.3, 0.7, 0.95])))

for depth in (1, 2, 50, 99):                              # cây lệch trái, lệch phải, zig-zag (sâu tối đa gần 100)
    left = TreeNode(0)
    cur = left
    for i in range(1, depth):
        cur.left = TreeNode(i)
        cur = cur.left
    check_tree(left)
    right = TreeNode(0)
    cur = right
    for i in range(1, depth):
        cur.right = TreeNode(i)
        cur = cur.right
    check_tree(right)
    zig = TreeNode(0)
    cur = zig
    for i in range(1, depth):
        if i % 2:
            cur.left = TreeNode(i)
            cur = cur.left
        else:
            cur.right = TreeNode(i)
            cur = cur.right
    check_tree(zig)

# Cây đầy đủ 63 nút (6 tầng)
check(list(range(63)))
# build_tree <-> tree_to_list khứ hồi
for _ in range(200):
    root = random_tree(rnd.randint(0, 50), rnd)
    vals = tree_to_list(root)
    assert tree_to_list(build_tree(vals)) == vals
print("PASS 1000 cây ngẫu nhiên, cây lệch/zig-zag, cây đầy đủ 63 nút, build_tree khứ hồi")

PASS 1000 cây ngẫu nhiên, cây lệch/zig-zag, cây đầy đủ 63 nút, build_tree khứ hồi


## 13. Các lỗi người mới hay gặp

### Lỗi 1: DFS mà không sắp theo độ sâu

In [18]:
def sai_dfs_khong_sap(root):
    cols = defaultdict(list)

    def dfs(node, col):
        if node is None:
            return
        cols[col].append(node.val)                      # sai: thứ tự DFS không phải thứ tự trên -> dưới
        dfs(node.left, col - 1)
        dfs(node.right, col + 1)

    dfs(root, 0)
    return [cols[c] for c in sorted(cols)]


v3, e3 = examples[2]
print("DFS không sắp độ sâu ->", sai_dfs_khong_sap(build_tree(v3)), "| đúng:", e3)
assert sai_dfs_khong_sap(build_tree(v3)) != e3

DFS không sắp độ sâu -> [[4], [9, 5], [3, 0, 1], [2, 8], [7]] | đúng: [[4], [9, 5], [3, 0, 1], [8, 2], [7]]


**Vì sao sai:** DFS tiền tự đi sâu hết nhánh trái trước nên gặp nút ở hàng sâu của nhánh trái trước các nút ở hàng nông của nhánh phải. Ở cột `1`, `2` (hàng 3, nhánh trái) bị đặt trước `8` (hàng 1). Phải ghi kèm độ sâu rồi sắp, hoặc dùng BFS.

### Lỗi 2: Sắp theo giá trị khi cùng cột (nhầm với bài 987)

In [19]:
def sai_sap_theo_gia_tri(root):
    cols = defaultdict(list)
    q = deque([(root, 0, 0)] if root else [])
    while q:
        node, c, r = q.popleft()
        cols[c].append((r, node.val))
        if node.left:
            q.append((node.left, c - 1, r + 1))
        if node.right:
            q.append((node.right, c + 1, r + 1))
    return [[v for _, v in sorted(cols[c])] for c in sorted(cols)]      # sai: sắp (hàng, GIÁ TRỊ)


vals = [1, 2, 3, 4, 6, 5, 7]
print("sắp theo giá trị ->", sai_sap_theo_gia_tri(build_tree(vals)), "| đúng:", [[4], [2], [1, 6, 5], [3], [7]])
assert sai_sap_theo_gia_tri(build_tree(vals)) != [[4], [2], [1, 6, 5], [3], [7]]

sắp theo giá trị -> [[4], [2], [1, 5, 6], [3], [7]] | đúng: [[4], [2], [1, 6, 5], [3], [7]]


**Vì sao sai:** bài này yêu cầu cùng hàng, cùng cột thì **trái trước** (`6` rồi `5`). Bài 987 mới sắp theo giá trị (`5` rồi `6`). Đọc kỹ quy tắc phân xử.

### Lỗi 3: Quên sắp các cột (dict giữ thứ tự chèn, không phải thứ tự cột)

In [20]:
def sai_khong_sap_cot(root):
    cols = {}
    q = deque([(root, 0)] if root else [])
    while q:
        node, c = q.popleft()
        cols.setdefault(c, []).append(node.val)
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    return list(cols.values())                          # sai: thứ tự chèn (0, -1, 1, ...) không phải -1, 0, 1


v1, e1 = examples[0]
print("không sắp cột ->", sai_khong_sap_cot(build_tree(v1)), "| đúng:", e1)
assert sai_khong_sap_cot(build_tree(v1)) != e1

không sắp cột -> [[3, 15], [9], [20], [7]] | đúng: [[9], [3, 15], [20], [7]]


**Vì sao sai:** `dict` ghi nhớ thứ tự **chèn khóa**: gốc (cột 0) vào trước, rồi cột −1, rồi cột 1. Kết quả `[[3, 15], [9], [20], [7]]`. Phải đọc theo `sorted(cols)` hoặc `range(min, max + 1)`.

### Lỗi 4: Quên cây rỗng

In [21]:
def sai_cay_rong(root):
    cols = defaultdict(list)
    q = deque([(root, 0)])                              # sai: root có thể là None
    while q:
        node, c = q.popleft()
        cols[c].append(node.val)                         # AttributeError khi node là None
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    return [cols[c] for c in sorted(cols)]


try:
    sai_cay_rong(None)
    print("không lỗi (bất ngờ)")
except AttributeError as exc:
    print("cây rỗng ->", type(exc).__name__, ":", exc)

cây rỗng -> AttributeError : 'NoneType' object has no attribute 'val'


**Vì sao sai:** `root = None` (cây rỗng là test hợp lệ) làm `None.val` lỗi. Luôn bắt đầu bằng `if root is None: return []` hoặc khởi tạo hàng đợi rỗng khi `root` là `None`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Duyệt theo cột / theo hàng ngang của cây" → gán **cột** `±1`, nhóm bằng `dict`; **BFS** giữ thứ tự (hàng, trái → phải) miễn phí.
- DFS thì phải ghi kèm **độ sâu** rồi sắp ổn định; với 987 sắp theo `(hàng, giá trị)`.
- Đọc cột theo thứ tự: `sorted(cols)` hoặc `range(min, max + 1)` (cột liên tiếp nên không có "khoảng trống").
- Dùng `deque` cho BFS (`popleft` `O(1)`), không dùng `list.pop(0)`.
- Luôn xử lý `root is None`.
- Khi test cây: viết `build_tree` từ danh sách tầng và `tree_to_list` để kiểm tra khứ hồi; oracle `(cột, hàng, vị trí)` không dựa vào BFS.

## 15. Biến thể của bài

1. **Vertical Order Traversal (LeetCode 987):** cùng hàng, cùng cột thì sắp theo giá trị tăng.
2. **Top view / Bottom view** của cây.
3. **Tổng các nút theo từng cột** (vertical sum).
4. **Chiều rộng của cây** (số cột).
5. **Binary Tree Level Order Traversal (LeetCode 102)** / **Right Side View (199).**

In [22]:
def vertical_order_987(root):
    """Biến thể 1: cùng (cột, hàng) thì sắp theo giá trị."""
    items = []

    def dfs(node, col, row):
        if node is None:
            return
        items.append((col, row, node.val))
        dfs(node.left, col - 1, row + 1)
        dfs(node.right, col + 1, row + 1)

    dfs(root, 0, 0)
    items.sort()
    result, last = [], None
    for col, row, val in items:
        if col != last:
            result.append([])
            last = col
        result[-1].append(val)
    return result


def top_view(root):
    """Biến thể 2: nút đầu tiên của mỗi cột (nhìn từ trên xuống)."""
    return [col[0] for col in vertical_order_bfs_dict(root)]


def bottom_view(root):
    """Biến thể 2b: nút cuối của mỗi cột trong thứ tự BFS (nhìn từ dưới lên, phải thắng khi cùng hàng)."""
    return [col[-1] for col in vertical_order_bfs_dict(root)]


def vertical_sums(root):
    """Biến thể 3."""
    return [sum(col) for col in vertical_order_bfs_dict(root)]


def tree_width_columns(root):
    """Biến thể 4: số cột."""
    return len(vertical_order_bfs_dict(root))


def level_order(root):
    """Biến thể 5: duyệt từng tầng."""
    result, q = [], deque([root] if root else [])
    while q:
        result.append([])
        for _ in range(len(q)):
            node = q.popleft()
            result[-1].append(node.val)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    return result


r = build_tree([3, 9, 20, None, None, 15, 7])
assert top_view(r) == [9, 3, 20, 7]
assert vertical_sums(r) == [9, 18, 20, 7]
assert tree_width_columns(r) == 4
assert level_order(r) == [[3], [9, 20], [15, 7]]
assert vertical_order_987(build_tree([1, 2, 3, 4, 6, 5, 7])) == [[4], [2], [1, 5, 6], [3], [7]]
rnd = random.Random(15)
for _ in range(200):
    root = random_tree(rnd.randint(0, 30), rnd)
    ref = vertical_order_brute(root)
    assert [sorted(c) for c in vertical_order_987(root)] == [sorted(c) for c in ref]
    assert vertical_sums(root) == [sum(c) for c in ref]
    assert sorted(v for lvl in level_order(root) for v in lvl) == sorted(v for c in ref for v in c)
print("PASS biến thể: 987, top view, tổng cột, số cột, level order")

PASS biến thể: 987, top view, tổng cột, số cột, level order


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `(cột, hàng, vị trí)` rồi sắp | oracle độc lập | `O(n log n)` | `O(n)` | kiểm thử |
| 2. BFS + dict | `cols[cột].append` | `O(n + w log w)` | `O(n)` | **cách nộp bài** |
| 3. DFS + sắp theo độ sâu | khóa độ sâu, sắp ổn định | `O(n log n)` | `O(n + h)` | khi quen DFS |
| 4. BFS hai lượt | mảng theo `min..max` cột | `O(n)` | `O(n)` | tránh sắp xếp |
| 5. BFS theo tầng | tách tầng, `range(min, max + 1)` | `O(n)` | `O(n)` | mở rộng sang 987 |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `vertical_order_bfs_dict(build_tree([1, 2, 3]))` và `vertical_order_bfs_dict(None)`.

**Bài 2 (tính tay).** Với cây `[1, 2, 3, None, 4, 5]`, gán cột cho từng nút rồi viết kết quả.

**Bài 3 (điền TODO).** Hoàn thành `vertical_order_todo` theo Cách 2 (BFS + dict).

In [23]:
def vertical_order_todo(root):
    # TODO 1: nếu root là None trả []
    # TODO 2: cols = defaultdict(list); q = deque([(root, 0)])
    # TODO 3: lấy (node, c) ra; cols[c].append(node.val)
    # TODO 4: thêm con trái với cột c - 1, con phải với c + 1 (nếu có)
    # TODO 5: trả [cols[c] for c in sorted(cols)]
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `vertical_order_dfs2(root)` bằng DFS **lặp** (ngăn xếp), ghi `(độ sâu, thứ tự gặp, giá trị)` vào từng cột, rồi sắp theo `(độ sâu, thứ tự gặp)`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `top_view2(root)` (nhìn từ trên xuống) không dùng `vertical_order_*`, chỉ dùng BFS + dict `col → giá trị đầu tiên`. (b) Viết `vertical_sums2(root)` trả tổng từng cột.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: con trái ở cột −1, con phải ở cột +1; cây rỗng trả `[]`. Bài 2: tính cột từng nút từ gốc xuống.
- **Mức 2 (vừa):** Bài 1: `[[2], [1], [3]]` và `[]`. Bài 2: cột của `1, 2, 3, 4, 5` là `0, −1, 1, 0, 0`; kết quả `[[2], [1, 4, 5], [3]]`. Bài 3: năm TODO trong comment.
- **Mức 3 (sâu):** Bài 4: ngăn xếp `(node, col, depth)`, đẩy **phải trước trái** để lấy ra trái trước; thứ tự gặp là một bộ đếm tăng dần. Bài 5(a): BFS, dùng `setdefault(c, node.val)` để chỉ giữ giá trị đầu tiên; (b) cộng dồn `sums[c] += val`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert vertical_order_bfs_dict(build_tree([1, 2, 3])) == [[2], [1], [3]]
assert vertical_order_bfs_dict(None) == []

# Bài 2
assert vertical_order_bfs_dict(build_tree([1, 2, 3, None, 4, 5])) == [[2], [1, 4, 5], [3]]

# Bài 3
def vertical_order_todo_ans(root):
    if root is None:
        return []
    cols = defaultdict(list)
    q = deque([(root, 0)])
    while q:
        node, c = q.popleft()
        cols[c].append(node.val)
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    return [cols[c] for c in sorted(cols)]


# Bài 4
def vertical_order_dfs2(root):
    if root is None:
        return []
    cols = defaultdict(list)
    stack = [(root, 0, 0)]
    seen = 0
    while stack:
        node, col, depth = stack.pop()
        cols[col].append((depth, seen, node.val))
        seen += 1
        if node.right:
            stack.append((node.right, col + 1, depth + 1))      # đẩy phải trước để trái được lấy ra trước
        if node.left:
            stack.append((node.left, col - 1, depth + 1))
    return [[v for _, _, v in sorted(cols[c])] for c in sorted(cols)]


# Bài 5
def top_view2(root):
    if root is None:
        return []
    first = {}
    q = deque([(root, 0)])
    while q:
        node, c = q.popleft()
        first.setdefault(c, node.val)
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    return [first[c] for c in sorted(first)]


def vertical_sums2(root):
    if root is None:
        return []
    sums = defaultdict(int)
    q = deque([(root, 0)])
    while q:
        node, c = q.popleft()
        sums[c] += node.val
        if node.left:
            q.append((node.left, c - 1))
        if node.right:
            q.append((node.right, c + 1))
    return [sums[c] for c in sorted(sums)]


rnd = random.Random(19)
for _ in range(300):
    root = random_tree(rnd.randint(0, 40), rnd, skew=rnd.choice([0, 0.5]))
    base = vertical_order_brute(root)
    assert vertical_order_todo_ans(root) == base
    assert vertical_order_dfs2(root) == base
    assert top_view2(root) == [c[0] for c in base]
    assert vertical_sums2(root) == [sum(c) for c in base]
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (BFS + dict theo cột)

```python
class Solution:
    def verticalOrder(self, root: Optional[TreeNode]) -> List[List[int]]:
        if root is None:
            return []
        cols = defaultdict(list)
        q = deque([(root, 0)])
        while q:
            node, c = q.popleft()
            cols[c].append(node.val)
            if node.left:
                q.append((node.left, c - 1))
            if node.right:
                q.append((node.right, c + 1))
        return [cols[c] for c in sorted(cols)]
```

**Checklist**

- [ ] Đã xử lý `root is None` (trả `[]`)?
- [ ] Gốc ở cột `0`; trái `c − 1`, phải `c + 1`?
- [ ] Dùng **BFS** (hoặc DFS có sắp theo độ sâu) để đúng thứ tự trên → dưới?
- [ ] Thêm con **trái trước phải** vào hàng đợi (để cùng hàng thì trái trước)?
- [ ] Đọc các cột theo `sorted(cols)` (không dùng thứ tự chèn của dict)?
- [ ] Phân biệt với 987 (cùng hàng cùng cột sắp theo **giá trị**)?
- [ ] Đã thử: cây rỗng, một nút, cây lệch, giá trị trùng, nhiều nút cùng cột cùng hàng?
- [ ] Đã đối chiếu với oracle `(cột, hàng, vị trí)` trên cây ngẫu nhiên có seed?
- [ ] Skeleton đúng `class Solution` / `verticalOrder(self, root: Optional[TreeNode]) -> List[List[int]]`?